In [1]:
import sys
import os

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Current working directory:", os.getcwd())

Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Python executable: /usr/bin/python3
Current working directory: /content


In [2]:
%cd /content

!git clone https://github.com/BasanthNalla/freshcart-pipelines.git

/content
Cloning into 'freshcart-pipelines'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 44 (delta 6), reused 42 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 52.70 KiB | 4.39 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [3]:
%cd /content/freshcart-pipelines

!ls -la
!find iris_project -maxdepth 3 -type f

/content/freshcart-pipelines
total 40
drwxr-xr-x 7 root root 4096 Nov 16 11:44 .
drwxr-xr-x 1 root root 4096 Nov 16 11:44 ..
drwxr-xr-x 3 root root 4096 Nov 16 11:44 elt
drwxr-xr-x 8 root root 4096 Nov 16 11:44 .git
drwxr-xr-x 3 root root 4096 Nov 16 11:44 .github
-rw-r--r-- 1 root root  206 Nov 16 11:44 .gitignore
drwxr-xr-x 3 root root 4096 Nov 16 11:44 iris_project
-rw-r--r-- 1 root root  512 Nov 16 11:44 README.md
-rw-r--r-- 1 root root   64 Nov 16 11:44 requirements.txt
drwxr-xr-x 2 root root 4096 Nov 16 11:44 tests
iris_project/__init__.py
iris_project/transformers/__init__.py
iris_project/transformers/prepare_data.py


In [4]:
!cat iris_project/transformers/prepare_data.py


from sklearn.model_selection import train_test_split

# Mage decorator compatibility
try:
    from mage_ai.data_preparation.decorators import transformer
except ImportError:
    def transformer(func):
        return func


@transformer
def transform(df, *args, **kwargs):
    """
    Split the Iris dataset into 80% training
    data and 20% testing data.
    """

    train_indices, test_indices = train_test_split(
        df.index,
        test_size=0.2,
        random_state=42,
        stratify=df["target"]
    )

    output = df.copy()

    output["split"] = "train"
    output.loc[test_indices, "split"] = "test"

    return output


In [5]:
%%writefile iris_project/transformers/prepare_data.py

from sklearn.model_selection import train_test_split

# Mage decorator compatibility
try:
    from mage_ai.data_preparation.decorators import transformer, test
except ImportError:
    def transformer(func):
        return func

    def test(func):
        return func


@transformer
def transform(df, *args, **kwargs):
    """
    Split the Iris dataset into 80% training
    data and 20% testing data.
    """

    train_indices, test_indices = train_test_split(
        df.index,
        test_size=0.2,
        random_state=42,
        stratify=df["target"]
    )

    output = df.copy()

    output["split"] = "train"
    output.loc[test_indices, "split"] = "test"

    return output


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 150

    assert (output["split"] == "train").sum() == 120
    assert (output["split"] == "test").sum() == 30

Overwriting iris_project/transformers/prepare_data.py


In [6]:
!cat iris_project/transformers/prepare_data.py


from sklearn.model_selection import train_test_split

# Mage decorator compatibility
try:
    from mage_ai.data_preparation.decorators import transformer, test
except ImportError:
    def transformer(func):
        return func

    def test(func):
        return func


@transformer
def transform(df, *args, **kwargs):
    """
    Split the Iris dataset into 80% training
    data and 20% testing data.
    """

    train_indices, test_indices = train_test_split(
        df.index,
        test_size=0.2,
        random_state=42,
        stratify=df["target"]
    )

    output = df.copy()

    output["split"] = "train"
    output.loc[test_indices, "split"] = "test"

    return output


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 150

    assert (output["split"] == "train").sum() == 120
    assert (output["split"] == "test").sum() == 30


In [7]:
!cat requirements.txt

pandas
numpy
scikit-learn
pytest
pika
snowflake-connector-python

In [8]:
!python --version

Python 3.13.18


In [9]:
!pip install uv -q

In [10]:
!uv venv /content/mage_env --python 3.13

Using CPython 3.13.18 interpreter at: /usr/bin/python3
Creating virtual environment at: /content/mage_env
Activate with: source /content/mage_env/bin/activate


In [11]:
!uv pip install --python /content/mage_env/bin/python mage-ai

Using Python 3.13.18 environment at: /content/mage_env
Resolved 182 packages in 2.35s
error: Failed to build `pillow==10.3.0`
  cause: The build backend returned an error
  cause: Call to `backend.get_requires_for_build_wheel` failed (exit status: 1)

         [stderr]
         Traceback (most recent call last):
           File "<string>", line 14, in <module>
             requires = get_requires_for_build({})
           File "/root/.cache/uv/builds-v0/.tmpHQpWoJ/lib/python3.13/site-packages/setuptools/build_meta.py", line 333, in get_requires_for_build_wheel
             return self._get_build_requires(config_settings, requirements=[])
                    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
           File "/root/.cache/uv/builds-v0/.tmpHQpWoJ/lib/python3.13/site-packages/setuptools/build_meta.py", line 301, in _get_build_requires
             self.run_setup()
             ~~~~~~~~~~~~~~^^
           File "/root/.cache/uv/builds-v0/.tmpHQpWoJ/lib/python3.13/site

In [12]:
!uv python install 3.12

Installed Python 3.12.16 in 2.90s
 + cpython-3.12.16-linux-x86_64-gnu (python3.12)


In [13]:
!rm -rf /content/mage_env

In [14]:
!uv venv /content/mage_env --python 3.12

Using CPython 3.12.16
Creating virtual environment at: /content/mage_env
Activate with: source /content/mage_env/bin/activate


In [15]:
!/content/mage_env/bin/python --version

Python 3.12.16


In [16]:
!uv pip install --python /content/mage_env/bin/python mage-ai

Using Python 3.12.16 environment at: /content/mage_env
Resolved 182 packages in 1.29s
Prepared 26 packages in 10.12s
Installed 182 packages in 837ms
 + aiofiles==22.1.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.10.0
 + aiosignal==1.4.0
 + alembic==1.14.1
 + altair==4.2.2
 + annotated-types==0.8.0
 + anyio==4.14.2
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==26.2.0
 + arrow==1.4.0
 + asttokens==3.0.2
 + async-lru==2.4.0
 + attrs==26.1.0
 + babel==2.18.0
 + backcall==0.2.0
 + bcrypt==4.0.1
 + beautifulsoup4==4.15.0
 + bleach==6.4.0
 + cachetools==8.0.0
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.1.8
 + cloudpickle==3.1.2
 + colorama==0.4.6
 + comm==0.2.3
 + cron-converter==1.2.1
 + cryptography==42.0.4
 + dask==2026.8.0
 + datadog==0.44.0
 + debugpy==1.8.22
 + decorator==5.3.1
 + defusedxml==0.7.1
 + deprecated==3.0.1
 + entrypoints==0.4.2
 + executing==2.3.0
 + faker==4.14.0
 + fastjsonschema==2.22.3
 + fqdn==1.6.0
 + freezegun==1.2.2
 + frozenlist==

In [17]:
!/content/mage_env/bin/mage --help

                                                                                
 Usage: mage [OPTIONS] COMMAND [ARGS]...                                        
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          Install completion for the current shell.      │
│ --show-completion             Show completion for the current shell, to copy │
│                               it or customize the installation.              │
│ --help                        Show this message and exit.                    │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ init                           Initialize Mage project.                      │
│ start                          Start Mage server and UI.                     │
│ run                       

In [18]:
!uv pip install --python /content/mage_env/bin/python pandas numpy scikit-learn

Using Python 3.12.16 environment at: /content/mage_env
Checked 3 packages in 5ms


In [19]:
%cd /content/freshcart-pipelines

import os

folders = [
    "iris_project/data_loaders",
    "iris_project/transformers",
    "iris_project/data_exporters"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("All project folders are ready.")

/content/freshcart-pipelines
All project folders are ready.


In [20]:
from pathlib import Path

files = [
    "iris_project/__init__.py",
    "iris_project/data_loaders/__init__.py",
    "iris_project/transformers/__init__.py",
    "iris_project/data_exporters/__init__.py"
]

for file in files:
    Path(file).touch()

print("Initialization files created.")

Initialization files created.


**Create load_iris.py**

In [21]:

%%writefile iris_project/data_loaders/load_iris.py

from sklearn.datasets import load_iris

# Mage decorator compatibility
try:
    from mage_ai.data_preparation.decorators import data_loader, test
except ImportError:
    def data_loader(func):
        return func

    def test(func):
        return func


@data_loader
def load_data(*args, **kwargs):
    """
    Load the Iris dataset as a Pandas DataFrame.
    """
    iris = load_iris(as_frame=True)
    return iris.frame


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 150
    assert "target" in output.columns
    assert output.shape[1] == 5
    assert output.isnull().sum().sum() == 0


Writing iris_project/data_loaders/load_iris.py


In [22]:
!cat iris_project/data_loaders/load_iris.py


from sklearn.datasets import load_iris

# Mage decorator compatibility
try:
    from mage_ai.data_preparation.decorators import data_loader, test
except ImportError:
    def data_loader(func):
        return func

    def test(func):
        return func


@data_loader
def load_data(*args, **kwargs):
    """
    Load the Iris dataset as a Pandas DataFrame.
    """
    iris = load_iris(as_frame=True)
    return iris.frame


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 150
    assert "target" in output.columns
    assert output.shape[1] == 5
    assert output.isnull().sum().sum() == 0


In [23]:
import sys

sys.path.insert(0, "/content/freshcart-pipelines")

from iris_project.data_loaders.load_iris import load_data

df = load_data()

print("Dataset shape:", df.shape)
print("\nFirst five records:")
display(df.head())

print("\nTarget distribution:")
print(df["target"].value_counts().sort_index())

Dataset shape: (150, 5)

First five records:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0



Target distribution:
target
0    50
1    50
2    50
Name: count, dtype: int64


**Create train_model.py**

In [24]:

%%writefile iris_project/transformers/train_model.py

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Mage decorator compatibility
try:
    from mage_ai.data_preparation.decorators import transformer, test
except ImportError:
    def transformer(func):
        return func

    def test(func):
        return func


@transformer
def train_model(df, *args, **kwargs):
    """
    Train a Decision Tree classifier and calculate test accuracy.
    """

    train_df = df[df["split"] == "train"]
    test_df = df[df["split"] == "test"]

    feature_columns = [
        "sepal length (cm)",
        "sepal width (cm)",
        "petal length (cm)",
        "petal width (cm)"
    ]

    X_train = train_df[feature_columns]
    y_train = train_df["target"]

    X_test = test_df[feature_columns]
    y_test = test_df["target"]

    max_depth = kwargs.get("max_depth", 3)

    model = DecisionTreeClassifier(
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    return {
        "model": model,
        "predictions": predictions.tolist(),
        "y_test": y_test.tolist(),
        "accuracy": accuracy,
        "max_depth": max_depth
    }


@test
def test_model_accuracy(output, *args):
    assert output is not None
    assert 0 <= output["accuracy"] <= 1
    assert output["accuracy"] >= 0.85

Writing iris_project/transformers/train_model.py


In [25]:
!cat iris_project/transformers/train_model.py


from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Mage decorator compatibility
try:
    from mage_ai.data_preparation.decorators import transformer, test
except ImportError:
    def transformer(func):
        return func

    def test(func):
        return func


@transformer
def train_model(df, *args, **kwargs):
    """
    Train a Decision Tree classifier and calculate test accuracy.
    """

    train_df = df[df["split"] == "train"]
    test_df = df[df["split"] == "test"]

    feature_columns = [
        "sepal length (cm)",
        "sepal width (cm)",
        "petal length (cm)",
        "petal width (cm)"
    ]

    X_train = train_df[feature_columns]
    y_train = train_df["target"]

    X_test = test_df[feature_columns]
    y_test = test_df["target"]

    max_depth = kwargs.get("max_depth", 3)

    model = DecisionTreeClassifier(
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictio

In [26]:
from iris_project.transformers.train_model import train_model

model_output = train_model(
    __import__("iris_project.transformers.prepare_data",
               fromlist=["transform"]).transform(df),
    max_depth=3
)

print("Maximum depth:", model_output["max_depth"])
print("Test accuracy:", model_output["accuracy"])
print("Number of predictions:", len(model_output["predictions"]))

Maximum depth: 3
Test accuracy: 0.9666666666666667
Number of predictions: 30


**Create save_results.py**

In [27]:

%%writefile iris_project/data_exporters/save_results.py

import pandas as pd
from pathlib import Path

# Mage decorator compatibility
try:
    from mage_ai.data_preparation.decorators import data_exporter, test
except ImportError:
    def data_exporter(func):
        return func

    def test(func):
        return func


@data_exporter
def save_results(model_output, *args, **kwargs):
    """
    Save model predictions and actual values to a CSV file.
    """

    output_df = pd.DataFrame({
        "actual": model_output["y_test"],
        "predicted": model_output["predictions"]
    })

    output_path = Path("/content/freshcart-pipelines/iris_project/iris_predictions.csv")

    output_df.to_csv(output_path, index=False)

    return {
        "file_path": str(output_path),
        "rows_saved": len(output_df),
        "accuracy": model_output["accuracy"]
    }


@test
def test_output(output, *args):
    assert output is not None
    assert output["rows_saved"] == 30
    assert Path(output["file_path"]).exists()


Writing iris_project/data_exporters/save_results.py


In [28]:
!cat iris_project/data_exporters/save_results.py


import pandas as pd
from pathlib import Path

# Mage decorator compatibility
try:
    from mage_ai.data_preparation.decorators import data_exporter, test
except ImportError:
    def data_exporter(func):
        return func

    def test(func):
        return func


@data_exporter
def save_results(model_output, *args, **kwargs):
    """
    Save model predictions and actual values to a CSV file.
    """

    output_df = pd.DataFrame({
        "actual": model_output["y_test"],
        "predicted": model_output["predictions"]
    })

    output_path = Path("/content/freshcart-pipelines/iris_project/iris_predictions.csv")

    output_df.to_csv(output_path, index=False)

    return {
        "file_path": str(output_path),
        "rows_saved": len(output_df),
        "accuracy": model_output["accuracy"]
    }


@test
def test_output(output, *args):
    assert output is not None
    assert output["rows_saved"] == 30
    assert Path(output["file_path"]).exists()


In [29]:
from iris_project.data_exporters.save_results import save_results

save_output = save_results(model_output)

print("File path:", save_output["file_path"])
print("Rows saved:", save_output["rows_saved"])
print("Accuracy:", save_output["accuracy"])

File path: /content/freshcart-pipelines/iris_project/iris_predictions.csv
Rows saved: 30
Accuracy: 0.9666666666666667


In [30]:
import pandas as pd

predictions_df = pd.read_csv(
    "/content/freshcart-pipelines/iris_project/iris_predictions.csv"
)

display(predictions_df.head())
print("Total rows:", len(predictions_df))

,actual,predicted
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


Total rows: 30


**Create metadata.yaml**

In [34]:
%%writefile iris_project/metadata.yaml

blocks:
- uuid: load_iris
  type: data_loader
  name: load_iris
  language: python
  executor_type: local_python
  upstream_blocks: []
  downstream_blocks:
  - prepare_data
  configuration: {}

- uuid: prepare_data
  type: transformer
  name: prepare_data
  language: python
  executor_type: local_python
  upstream_blocks:
  - load_iris
  downstream_blocks:
  - train_model
  configuration: {}

- uuid: train_model
  type: transformer
  name: train_model
  language: python
  executor_type: local_python
  upstream_blocks:
  - prepare_data
  downstream_blocks:
  - save_results
  configuration: {}

- uuid: save_results
  type: data_exporter
  name: save_results
  language: python
  executor_type: local_python
  upstream_blocks:
  - train_model
  downstream_blocks: []
  configuration: {}

name: iris_classifier
type: python

Overwriting iris_project/metadata.yaml


In [35]:
!cat iris_project/metadata.yaml


blocks:
- uuid: load_iris
  type: data_loader
  name: load_iris
  language: python
  executor_type: local_python
  upstream_blocks: []
  downstream_blocks:
  - prepare_data
  configuration: {}

- uuid: prepare_data
  type: transformer
  name: prepare_data
  language: python
  executor_type: local_python
  upstream_blocks:
  - load_iris
  downstream_blocks:
  - train_model
  configuration: {}

- uuid: train_model
  type: transformer
  name: train_model
  language: python
  executor_type: local_python
  upstream_blocks:
  - prepare_data
  downstream_blocks:
  - save_results
  configuration: {}

- uuid: save_results
  type: data_exporter
  name: save_results
  language: python
  executor_type: local_python
  upstream_blocks:
  - train_model
  downstream_blocks: []
  configuration: {}

name: iris_classifier
type: python


In [36]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage run iris_project iris_classifier --test

/content/freshcart-pipelines
DBT library not installed.
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /content/mage_env/lib/python3.12/site-packages/mage_ai/cli/main.py:235 in    │
│ run                                                                          │
│                                                                              │
│   232 │   │   # Initialize db_connection session before getting the pipeline │
│   233 │   │   # "mage_secret_var" syntax is used in the project's metadata.y │
│   234 │   │   db_connection.start_session()                                  │
│ ❱ 235 │   │   pipeline = Pipeline.get(pipeline_uuid, repo_path=project_path) │
│   236 │   │                                                                  │
│   237 │   │   if pipeline_run_id is None:                                    │
│   238 │   │   │   default_variables = get_global_variables(pipeline_uuid)    │
│                                                    

In [37]:
%cd /content/freshcart-pipelines

import os
import shutil

pipeline_dir = "iris_project/pipelines/iris_classifier"

os.makedirs(pipeline_dir, exist_ok=True)

shutil.move(
    "iris_project/metadata.yaml",
    f"{pipeline_dir}/metadata.yaml"
)

print("Pipeline metadata moved successfully!")

/content/freshcart-pipelines
Pipeline metadata moved successfully!


In [38]:
%%writefile iris_project/metadata.yaml

project_type: standalone

variables_dir: ~/.mage_data

variables_retention_period: '90d'

Writing iris_project/metadata.yaml


In [39]:
!find iris_project -maxdepth 4 -type f

iris_project/metadata.yaml
iris_project/__init__.py
iris_project/data_loaders/__init__.py
iris_project/data_loaders/__pycache__/__init__.cpython-313.pyc
iris_project/data_loaders/__pycache__/load_iris.cpython-313.pyc
iris_project/data_loaders/load_iris.py
iris_project/pipelines/iris_classifier/metadata.yaml
iris_project/data_exporters/save_results.py
iris_project/data_exporters/__init__.py
iris_project/data_exporters/__pycache__/save_results.cpython-313.pyc
iris_project/data_exporters/__pycache__/__init__.cpython-313.pyc
iris_project/iris_predictions.csv
iris_project/__pycache__/__init__.cpython-313.pyc
iris_project/transformers/__init__.py
iris_project/transformers/prepare_data.py
iris_project/transformers/train_model.py
iris_project/transformers/__pycache__/prepare_data.cpython-313.pyc
iris_project/transformers/__pycache__/train_model.cpython-313.pyc
iris_project/transformers/__pycache__/__init__.cpython-313.pyc


In [40]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage run iris_project iris_classifier --test

/content/freshcart-pipelines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /content/mage_env/lib/python3.12/site-packages/mage_ai/data_preparation/exec │
│ utors/pipeline_executor.py:82 in execute                                     │
│                                                                              │
│    79 │   │   │   │   )                                                      │
│    80 │   │   # Execute the task based on current context                    │
│    81 │   │   try:                                                           │
│ ❱  82 │   │   │   loop = asyncio.get_running_loop()                          │
│    83 │   │   │   logger.info(f'[PipelineExecutor] Found running loop {loop} │
│    84 │   │   │   # We're in an async context, use create_task               │
│    85 │   │   │   ta

In [41]:
%cd /content/freshcart-pipelines

!cat iris_project/data_loaders/load_iris.py

/content/freshcart-pipelines

from sklearn.datasets import load_iris

# Mage decorator compatibility
try:
    from mage_ai.data_preparation.decorators import data_loader, test
except ImportError:
    def data_loader(func):
        return func

    def test(func):
        return func


@data_loader
def load_data(*args, **kwargs):
    """
    Load the Iris dataset as a Pandas DataFrame.
    """
    iris = load_iris(as_frame=True)
    return iris.frame


@test
def test_output(output, *args):
    assert output is not None
    assert len(output) == 150
    assert "target" in output.columns
    assert output.shape[1] == 5
    assert output.isnull().sum().sum() == 0


In [42]:
!/content/mage_env/bin/python -c "from mage_ai.data_preparation.decorators import data_loader, test; print('Imports successful'); print(data_loader)"

Imports successful
<function data_loader at 0x79b01b919260>


In [43]:
!/content/mage_env/bin/python -c "import sys; print(sys.executable); print(sys.version)"

/content/mage_env/bin/python
3.12.16 (main, Nov 11 2026, 00:15:59) [Clang 23.1.3 ]


In [44]:
!find iris_project -maxdepth 4 -type f

iris_project/metadata.yaml
iris_project/__init__.py
iris_project/data_loaders/__init__.py
iris_project/data_loaders/__pycache__/__init__.cpython-313.pyc
iris_project/data_loaders/__pycache__/load_iris.cpython-313.pyc
iris_project/data_loaders/load_iris.py
iris_project/pipelines/iris_classifier/metadata.yaml
iris_project/data_exporters/save_results.py
iris_project/data_exporters/__init__.py
iris_project/data_exporters/__pycache__/save_results.cpython-313.pyc
iris_project/data_exporters/__pycache__/__init__.cpython-313.pyc
iris_project/iris_predictions.csv
iris_project/__pycache__/__init__.cpython-313.pyc
iris_project/transformers/__init__.py
iris_project/transformers/prepare_data.py
iris_project/transformers/train_model.py
iris_project/transformers/__pycache__/prepare_data.cpython-313.pyc
iris_project/transformers/__pycache__/train_model.cpython-313.pyc
iris_project/transformers/__pycache__/__init__.cpython-313.pyc


In [45]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/python -c "from iris_project.data_loaders.load_iris import load_data; print('Function:', load_data); print('Attributes:', vars(load_data)); print('Module:', load_data.__module__)"

/content/freshcart-pipelines
Function: <function load_data at 0x7ca7752fed40>
Attributes: {}
Module: iris_project.data_loaders.load_iris


In [46]:
!/content/mage_env/bin/python -c "import inspect; from mage_ai.data_preparation.decorators import data_loader; print('Decorator:', data_loader); print('Decorator source:'); print(inspect.getsource(data_loader))"

Decorator: <function data_loader at 0x787459ef6700>
Decorator source:
def data_loader(function):
    return function



In [47]:
!/content/mage_env/bin/python -c "import inspect; from mage_ai.data_preparation.models.block import Block; print(inspect.getfile(Block))"

/content/mage_env/lib/python3.12/site-packages/mage_ai/data_preparation/models/block/__init__.py


In [48]:
!/content/mage_env/bin/python -c "from pathlib import Path; import mage_ai.data_preparation.models.block as b; print(Path(b.__file__).read_text()[:12000])"

import asyncio
import functools
import importlib.util
import inspect
import json
import logging
import os
import sys
import time
import traceback
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from datetime import datetime
from enum import Enum
from inspect import Parameter, isfunction, signature
from logging import Logger
from pathlib import Path
from queue import Queue
from typing import (
    Any,
    Callable,
    Dict,
    Generator,
    Iterable,
    List,
    Optional,
    Set,
    Tuple,
    Union,
)

import inflection
import pandas as pd
import polars as pl
import simplejson
import yaml
from jinja2 import Template

import mage_ai.data_preparation.decorators
from mage_ai.cache.block import BlockCache
from mage_ai.data.constants import InputDataType
from mage_ai.data.tabular.models import BatchSettings
from mage_ai.data_integrations.sources.constants import SQL_SOURCES_MAPPING
from mage_ai.data_preparation.logging.logger import DictLogger
from mage_ai.da

In [49]:
!/content/mage_env/bin/python -c "import inspect; from mage_ai.data_preparation.models.block import Block; print(inspect.getsource(Block.execute_sync))"

    def execute_sync(
        self,
        analyze_outputs: bool = False,
        block_run_outputs_cache: Dict[str, List] = None,
        build_block_output_stdout: Callable[..., object] = None,
        custom_code: str = None,
        execution_partition: str = None,
        from_notebook: bool = False,
        global_vars: Dict = None,
        logger: Logger = None,
        logging_tags: Dict = None,
        run_all_blocks: bool = False,
        update_status: bool = True,
        store_variables: bool = True,
        verify_output: bool = True,
        input_from_output: Dict = None,
        runtime_arguments: Dict = None,
        dynamic_block_index: int = None,
        dynamic_block_indexes: Dict = None,
        dynamic_block_uuid: str = None,
        dynamic_upstream_block_uuids: List[str] = None,
        run_settings: Dict = None,
        output_messages_to_logs: bool = False,
        disable_json_serialization: bool = False,
        data_integration_runtime_settings: Dict = N

In [50]:
!/content/mage_env/bin/python -c "import inspect; from mage_ai.data_preparation.models.block import Block; print(inspect.getsource(Block.execute_block))"

    def execute_block(
        self,
        block_run_outputs_cache: Dict[str, List] = None,
        build_block_output_stdout: Callable[..., object] = None,
        custom_code: str = None,
        execution_partition: str = None,
        from_notebook: bool = False,
        global_vars: Dict = None,
        input_args: List = None,
        logger: Logger = None,
        logging_tags: Dict = None,
        input_from_output: Dict = None,
        runtime_arguments: Dict = None,
        dynamic_block_index: int = None,
        dynamic_block_indexes: Dict = None,
        dynamic_upstream_block_uuids: List[str] = None,
        run_settings: Dict = None,
        data_integration_runtime_settings: str = None,
        execution_partition_previous: str = None,
        metadata: Dict = None,
        **kwargs,
    ) -> Dict:
        if logging_tags is None:
            logging_tags = dict()

        with self._redirect_streams(
            build_block_output_stdout=build_block_output_stdout,
  

In [51]:
!/content/mage_env/bin/python -c "import inspect; from mage_ai.data_preparation.models.block import Block; print(inspect.getsource(Block._execute_block))"

    def _execute_block(
        self,
        outputs_from_input_vars,
        custom_code: str = None,
        dynamic_block_index: int = None,
        dynamic_block_indexes: Dict = None,
        dynamic_upstream_block_uuids: List[str] = None,
        execution_partition: str = None,
        from_notebook: bool = False,
        global_vars: Dict = None,
        input_vars: List = None,
        logger: Logger = None,
        logging_tags: Dict = None,
        input_from_output: Dict = None,
        runtime_arguments: Dict = None,
        upstream_block_uuids: List[str] = None,
        run_settings: Dict = None,
        data_integration_runtime_settings: str = None,
        execution_partition_previous: str = None,
        **kwargs,
    ) -> List:
        if logging_tags is None:
            logging_tags = dict()

        if input_vars is None:
            input_vars = list()

        if self.get_data_integration_settings(
            dynamic_block_index=dynamic_block_index,
           

In [52]:
!/content/mage_env/bin/python -c "import inspect; from mage_ai.data_preparation.models.block import Block; print(inspect.getsource(Block._block_decorator))"

    def _block_decorator(self, decorated_functions) -> Callable:
        def custom_code(function) -> Callable:
            decorated_functions.append(function)
            return function

        return custom_code



In [53]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage run iris_project iris_classifier --test

/content/freshcart-pipelines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /content/mage_env/lib/python3.12/site-packages/mage_ai/data_preparation/exec │
│ utors/pipeline_executor.py:82 in execute                                     │
│                                                                              │
│    79 │   │   │   │   )                                                      │
│    80 │   │   # Execute the task based on current context                    │
│    81 │   │   try:                                                           │
│ ❱  82 │   │   │   loop = asyncio.get_running_loop()                          │
│    83 │   │   │   logger.info(f'[PipelineExecutor] Found running loop {loop} │
│    84 │   │   │   # 

In [54]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage run iris_project iris_classifier --test

/content/freshcart-pipelines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [55]:
import pandas as pd

df = pd.read_csv(
    "/content/freshcart-pipelines/iris_project/iris_predictions.csv"
)

print("Shape:", df.shape)
print("\nFirst 5 predictions:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

Shape: (30, 2)

First 5 predictions:


,actual,predicted
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0



Missing values:
actual       0
predicted    0
dtype: int64


In [56]:
!/content/mage_env/bin/mage run --help

                                                                                
 Usage: mage run [OPTIONS] PROJECT_PATH PIPELINE_UUID                           
                                                                                
 Run pipeline.                                                                  
                                                                                
╭─ Arguments ──────────────────────────────────────────────────────────────────╮
│ *    project_path       TEXT  path of the Mage project that contains the     │
│                               pipeline.                                      │
│                               [default: None]                                │
│                               [required]                                     │
│ *    pipeline_uuid      TEXT  uuid of the pipeline to be run.                │
│                               [default: None]                                │
│                           

In [57]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 1}'

/content/freshcart-pipelines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
FAIL: test_model_accuracy (block: train_model)
--------------------------------------------------------------
Traceback (most recent call last):
  File "/content/mage_env/lib/python3.12/site-packages/mage_ai/data_preparation/executors/pipeline_executor.py", line 82, in execute
    loop = asyncio.get_running_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: no running event loop

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/mage_env/lib/python3.12/site-packages/mage_ai/data_preparation/models/block/__init__.py", line 3350, in run_tests
    test_function(*outputs)
  File "<string>

In [58]:
from pathlib import Path

csv_path = Path("iris_project/iris_predictions.csv")
print("CSV created:", csv_path.exists())

CSV created: False


In [59]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 2}'

/content/freshcart-pipelines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [60]:
import pandas as pd
from pathlib import Path

csv_path = Path("iris_project/iris_predictions.csv")

print("CSV created:", csv_path.exists())

if csv_path.exists():
    df = pd.read_csv(csv_path)
    accuracy = (df["actual"] == df["predicted"]).mean()

    print("Rows:", len(df))
    print("Accuracy:", accuracy)
    print("Accuracy (%):", accuracy * 100)

CSV created: True
Rows: 30
Accuracy: 0.9333333333333333
Accuracy (%): 93.33333333333333


In [61]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 3}'

/content/freshcart-pipelines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [62]:
import pandas as pd
from pathlib import Path

csv_path = Path("iris_project/iris_predictions.csv")
print("CSV created:", csv_path.exists())

if csv_path.exists():
    df = pd.read_csv(csv_path)
    accuracy = (df["actual"] == df["predicted"]).mean()
    print("Accuracy:", accuracy)
    print("Accuracy (%):", accuracy * 100)

CSV created: True
Accuracy: 0.9666666666666667
Accuracy (%): 96.66666666666667


In [63]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 4}'

/content/freshcart-pipelines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [64]:
import pandas as pd
from pathlib import Path

csv_path = Path("iris_project/iris_predictions.csv")
print("CSV created:", csv_path.exists())

if csv_path.exists():
    df = pd.read_csv(csv_path)
    accuracy = (df["actual"] == df["predicted"]).mean()
    print("Accuracy:", accuracy)
    print("Accuracy (%):", accuracy * 100)

CSV created: True
Accuracy: 0.9333333333333333
Accuracy (%): 93.33333333333333


In [65]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 5}'

/content/freshcart-pipelines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [66]:
import pandas as pd
from pathlib import Path

csv_path = Path("iris_project/iris_predictions.csv")
print("CSV created:", csv_path.exists())

if csv_path.exists():
    df = pd.read_csv(csv_path)
    accuracy = (df["actual"] == df["predicted"]).mean()
    print("Accuracy:", accuracy)
    print("Accuracy (%):", accuracy * 100)

CSV created: True
Accuracy: 0.9333333333333333
Accuracy (%): 93.33333333333333


In [67]:
%cd /content/freshcart-pipelines

!cat iris_project/pipelines/iris_classifier/metadata.yaml

/content/freshcart-pipelines

blocks:
- uuid: load_iris
  type: data_loader
  name: load_iris
  language: python
  executor_type: local_python
  upstream_blocks: []
  downstream_blocks:
  - prepare_data
  configuration: {}

- uuid: prepare_data
  type: transformer
  name: prepare_data
  language: python
  executor_type: local_python
  upstream_blocks:
  - load_iris
  downstream_blocks:
  - train_model
  configuration: {}

- uuid: train_model
  type: transformer
  name: train_model
  language: python
  executor_type: local_python
  upstream_blocks:
  - prepare_data
  downstream_blocks:
  - save_results
  configuration: {}

- uuid: save_results
  type: data_exporter
  name: save_results
  language: python
  executor_type: local_python
  upstream_blocks:
  - train_model
  downstream_blocks: []
  configuration: {}

name: iris_classifier
type: python


In [68]:
!cat iris_project/pipelines/iris_classifier/metadata.yaml


blocks:
- uuid: load_iris
  type: data_loader
  name: load_iris
  language: python
  executor_type: local_python
  upstream_blocks: []
  downstream_blocks:
  - prepare_data
  configuration: {}

- uuid: prepare_data
  type: transformer
  name: prepare_data
  language: python
  executor_type: local_python
  upstream_blocks:
  - load_iris
  downstream_blocks:
  - train_model
  configuration: {}

- uuid: train_model
  type: transformer
  name: train_model
  language: python
  executor_type: local_python
  upstream_blocks:
  - prepare_data
  downstream_blocks:
  - save_results
  - evaluate_model
  configuration: {}

- uuid: save_results
  type: data_exporter
  name: save_results
  language: python
  executor_type: local_python
  upstream_blocks:
  - train_model
  downstream_blocks: []
  configuration: {}

- uuid: evaluate_model
  type: transformer
  name: evaluate_model
  language: python
  executor_type: local_python
  upstream_blocks:
  - train_model
  downstream_blocks: []
  configurati

In [69]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage run iris_project iris_classifier --test

/content/freshcart-pipelines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]

Evaluation Metrics:
Accuracy:  0.9667
Precision: 0.9697
Recall:    0.9667
F1 Score:  0.9666
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [70]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage start iris_project

/content/freshcart-pipelines
DBT library not installed.
Initializing global job manager:  <mage_ai.orchestration.job_manager.JobManager object at 0x79cc34af0380>
INFO:mage_ai.server.scheduler_manager:Start scheduler.
INFO:mage_ai.server.active_kernel:Switch active kernel: python3
INFO:alembic.runtime.migration:Context impl SQLiteImpl.
INFO:alembic.runtime.migration:Will assume non-transactional DDL.
INFO:alembic.runtime.migration:Running upgrade  -> c07a23ff782b, Add initial tables.
INFO:alembic.runtime.migration:Running upgrade c07a23ff782b -> 52ab80005742, Add variables to pipeline schedule.
INFO:alembic.runtime.migration:Running upgrade 52ab80005742 -> 053ee2c10d85, Add completed_at.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation

In [71]:
from google.colab import output

output.serve_kernel_port_as_window(6789)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [72]:
from google.colab import output

output.serve_kernel_port_as_iframe(6789, height=800)

<IPython.core.display.Javascript object>

In [73]:
from google.colab import output

url = output.eval_js("google.colab.kernel.proxyPort(6789)")
print(url)

https://6789-m-s-euw4b3-3qfjuxr52wul9-b.europe-west4.prod.r.colab.dev


In [74]:
!/content/mage_env/bin/mage start --help

                                                                                
 Usage: mage start [OPTIONS] [PROJECT_PATH]                                     
                                                                                
 Start Mage server and UI.                                                      
                                                                                
╭─ Arguments ──────────────────────────────────────────────────────────────────╮
│   project_path      [PROJECT_PATH]  path of the Mage project to be loaded.   │
│                                     [default: /content/freshcart-pipelines]  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --host                     TEXT  specify the host. [default: localhost]      │
│ --port                     TEXT  specify the port. [default: 6789]           │
│ --manage-instance         

In [77]:
%cd /content/freshcart-pipelines

!/content/mage_env/bin/mage start iris_project --host 0.0.0.0 --port 6789

/content/freshcart-pipelines
DBT library not installed.
Initializing global job manager:  <mage_ai.orchestration.job_manager.JobManager object at 0x7b43c4dbf1a0>
INFO:mage_ai.server.scheduler_manager:Start scheduler.
INFO:mage_ai.server.active_kernel:Switch active kernel: python3
INFO:alembic.runtime.migration:Context impl SQLiteImpl.
INFO:alembic.runtime.migration:Will assume non-transactional DDL.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Checking port 6789 on host 0.0.0.0...
INFO:mage_ai.server.server:Mage is running at http://0.0.0.0:6789 and serving project /content/freshcart-pipelines/iris_project
INFO:mage_ai.server.server:User authentication is enabled.
INFO:mage_ai.orchestration.pipeline_scheduler_original:Active pipe

In [76]:
from google.colab import output

url = output.eval_js("google.colab.kernel.proxyPort(6789)")
print(url)

https://6789-m-s-euw4b3-3qfjuxr52wul9-b.europe-west4.prod.r.colab.dev


In [78]:
import subprocess
import time

log_file = open("/content/mage_server.log", "w")

process = subprocess.Popen(
    [
        "/content/mage_env/bin/mage",
        "start",
        "iris_project",
        "--host", "0.0.0.0",
        "--port", "6789"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(10)

print("Mage process ID:", process.pid)
print("Running:", process.poll() is None)

Mage process ID: 24298
Running: True


In [79]:
!tail -30 /content/mage_server.log

DBT library not installed.
INFO:mage_ai.server.scheduler_manager:Start scheduler.
Initializing global job manager:  <mage_ai.orchestration.job_manager.JobManager object at 0x7bd413dfd2e0>
INFO:mage_ai.server.active_kernel:Switch active kernel: python3
INFO:alembic.runtime.migration:Context impl SQLiteImpl.
INFO:alembic.runtime.migration:Will assume non-transactional DDL.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
INFO:mage_ai.server.server:Mage is running at http://0.0.0.0:6789 and serving project /content/freshcart-pipelines/iris_project
INFO:mage_ai.server.server:User authentication is enabled.
INFO:mage_ai.orchestration.pipeline_scheduler_original:Active pipeline runs: []
0.00s - Debugger warning: It seems that frozen module

In [80]:
from google.colab import output

url = output.eval_js("google.colab.kernel.proxyPort(6789)")
print(url)

https://6789-m-s-euw4b3-3qfjuxr52wul9-b.europe-west4.prod.r.colab.dev


In [81]:
import requests

response = requests.get("http://127.0.0.1:6789/", timeout=10)

print("Status code:", response.status_code)
print("Response URL:", response.url)
print("Content type:", response.headers.get("content-type"))
print("Response preview:", response.text[:300])

Status code: 200
Response URL: http://127.0.0.1:6789/
Content type: text/html; charset=UTF-8
Response preview: <!DOCTYPE html><html lang="en"><head><meta charSet="utf-8"/><link href="/favicon.ico" rel="icon"/><title>Mage</title><meta content="width=device-width, initial-scale=1.0, maximum-scale=5.0, user-scalable=0" name="viewport"/><meta name="next-head-count" content="4"/><script>
// Polyfill for requestId


In [82]:
from google.colab import output

output.serve_kernel_port_as_iframe(6789, height=900)

<IPython.core.display.Javascript object>

In [92]:
%cd /content/freshcart-pipelines
!/content/mage_env/bin/mage run iris_project iris_classifier --test

/content/freshcart-pipelines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]

Evaluation Metrics:
Accuracy:  0.9667
Precision: 0.9697
Recall:    0.9667
F1 Score:  0.9666
Prediction Summary:
{'total_predictions': 30, 'correct_predictions': 29, 'incorrect_predictions': 1, 'accuracy': 0.9666666666666667}
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [93]:
%cd /content/freshcart-pipelines
!git status --short

/content/freshcart-pipelines
 M iris_project/transformers/prepare_data.py
?? iris_project/.ssh_tunnel/
?? iris_project/data_exporters/
?? iris_project/data_loaders/
?? iris_project/metadata.yaml
?? iris_project/pipelines/
?? iris_project/transformers/evaluate_model.py
?? iris_project/transformers/prediction_summary.py
?? iris_project/transformers/train_model.py


In [94]:
%cd /content/freshcart-pipelines

!cat .gitignore

/content/freshcart-pipelines
# Environment variables and secrets
.env

# Machine learning model files
*.pkl

# CSV files
*.csv

# Mage logs
mage.log

# Python cache
__pycache__/
*.pyc
.pytest_cache/

# Virtual environments
venv/
.venv/

In [95]:
with open(".gitignore", "a") as f:
    f.write("\n# Temporary tunnel files\niris_project/.ssh_tunnel/\n")

In [96]:
!git status --short

 M .gitignore
 M iris_project/transformers/prepare_data.py
?? iris_project/data_exporters/
?? iris_project/data_loaders/
?? iris_project/metadata.yaml
?? iris_project/pipelines/
?? iris_project/transformers/evaluate_model.py
?? iris_project/transformers/prediction_summary.py
?? iris_project/transformers/train_model.py


In [97]:
%cd /content/freshcart-pipelines
!pytest -v

/content/freshcart-pipelines
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/freshcart-pipelines
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
_________________ ERROR collecting tests/test_prepare_data.py __________________
ImportError while importing test module '/content/freshcart-pipelines/tests/test_prepare_data.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_prepare_data.py:3: in <module>
    from iris_

In [98]:
%cd /content/freshcart-pipelines

!ls -la
!ls -la iris_project
!ls -la tests

/content/freshcart-pipelines
total 44
drwxr-xr-x 8 root root 4096 Nov 16 13:27 .
drwxr-xr-x 1 root root 4096 Nov 16 13:07 ..
drwxr-xr-x 3 root root 4096 Nov 16 11:44 elt
drwxr-xr-x 8 root root 4096 Nov 16 13:25 .git
drwxr-xr-x 3 root root 4096 Nov 16 11:44 .github
-rw-r--r-- 1 root root  258 Nov 16 13:25 .gitignore
drwxr-xr-x 9 root root 4096 Nov 16 12:56 iris_project
drwxr-xr-x 3 root root 4096 Nov 16 13:27 .pytest_cache
-rw-r--r-- 1 root root  512 Nov 16 11:44 README.md
-rw-r--r-- 1 root root   64 Nov 16 11:44 requirements.txt
drwxr-xr-x 3 root root 4096 Nov 16 13:27 tests
total 44
drwxr-xr-x 9 root root 4096 Nov 16 12:56 .
drwxr-xr-x 8 root root 4096 Nov 16 13:27 ..
drwxr-xr-x 4 root root 4096 Nov 16 12:31 data_exporters
drwxr-xr-x 3 root root 4096 Nov 16 12:28 data_loaders
-rw-r--r-- 1 root root    0 Nov 16 11:56 __init__.py
drwxr-xr-x 2 root root 4096 Nov 16 12:36 .ipynb_checkpoints
-rw-r--r-- 1 root root  137 Nov 16 13:22 iris_predictions.csv
-rw-r--r-- 1 root root  136 Nov 16 12

In [99]:
%cd /content/freshcart-pipelines

!PYTHONPATH=/content/freshcart-pipelines pytest -v

/content/freshcart-pipelines
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/freshcart-pipelines
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 1 item                                                               

tests/test_prepare_data.py::test_split_is_80_20 PASSED                   [100%]

============================== 1 passed in 2.62s ===============================


In [100]:
%cd /content/freshcart-pipelines
!pytest -v

/content/freshcart-pipelines
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/freshcart-pipelines
configfile: pytest.ini
testpaths: tests
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 1 item                                                               

tests/test_prepare_data.py::test_split_is_80_20 PASSED                   [100%]

============================== 1 passed in 2.43s ===============================


In [101]:
%cd /content/freshcart-pipelines
!/content/mage_env/bin/mage run iris_project iris_classifier --test

/content/freshcart-pipelines
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Prediction Summary:
{'total_predictions': 30, 'correct_predictions': 29, 'incorrect_predictions': 1, 'accuracy': 0.9666666666666667}
Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]

Evaluation Metrics:
Accuracy:  0.9667
Precision: 0.9697
Recall:    0.9667
F1 Score:  0.9666
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.


In [102]:
%cd /content/freshcart-pipelines

!git status --short
!git diff --stat

/content/freshcart-pipelines
 M .gitignore
 M iris_project/transformers/prepare_data.py
?? iris_project/data_exporters/
?? iris_project/data_loaders/
?? iris_project/metadata.yaml
?? iris_project/pipelines/
?? iris_project/transformers/evaluate_model.py
?? iris_project/transformers/prediction_summary.py
?? iris_project/transformers/train_model.py
?? pytest.ini
 .gitignore                                |  4 +++-
 iris_project/transformers/prepare_data.py | 25 +++++++++++++++++--------
 2 files changed, 20 insertions(+), 9 deletions(-)


In [103]:
%cd /content/freshcart-pipelines

!git add .gitignore pytest.ini iris_project/
!git status --short

/content/freshcart-pipelines
M  .gitignore
A  iris_project/data_exporters/__init__.py
A  iris_project/data_exporters/save_results.py
A  iris_project/data_loaders/__init__.py
A  iris_project/data_loaders/load_iris.py
A  iris_project/metadata.yaml
A  iris_project/pipelines/iris_classifier/metadata.yaml
A  iris_project/transformers/evaluate_model.py
A  iris_project/transformers/prediction_summary.py
M  iris_project/transformers/prepare_data.py
A  iris_project/transformers/train_model.py
A  pytest.ini


In [104]:
!git diff --cached --stat

 .gitignore                                         |  4 +-
 iris_project/data_exporters/__init__.py            |  0
 iris_project/data_exporters/save_results.py        | 34 +++++++++++
 iris_project/data_loaders/__init__.py              |  0
 iris_project/data_loaders/load_iris.py             | 20 +++++++
 iris_project/metadata.yaml                         |  6 ++
 .../pipelines/iris_classifier/metadata.yaml        | 68 ++++++++++++++++++++++
 iris_project/transformers/evaluate_model.py        | 61 +++++++++++++++++++
 iris_project/transformers/prediction_summary.py    | 33 +++++++++++
 iris_project/transformers/prepare_data.py          | 25 +++++---
 iris_project/transformers/train_model.py           | 54 +++++++++++++++++
 pytest.ini                                         |  3 +
 12 files changed, 299 insertions(+), 9 deletions(-)


In [105]:
!git commit -m "Completed Mage Iris ML pipeline with evaluation and tests"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@a9b9256bc95b.(none)')


In [108]:
%cd /content/freshcart-pipelines
!git commit -m "Completed Mage Iris ML pipeline with evaluation and tests"

/content/freshcart-pipelines
[main 5aa45bd] Completed Mage Iris ML pipeline with evaluation and tests
 12 files changed, 299 insertions(+), 9 deletions(-)
 create mode 100644 iris_project/data_exporters/__init__.py
 create mode 100644 iris_project/data_exporters/save_results.py
 create mode 100644 iris_project/data_loaders/__init__.py
 create mode 100644 iris_project/data_loaders/load_iris.py
 create mode 100644 iris_project/metadata.yaml
 create mode 100644 iris_project/pipelines/iris_classifier/metadata.yaml
 create mode 100644 iris_project/transformers/evaluate_model.py
 create mode 100644 iris_project/transformers/prediction_summary.py
 create mode 100644 iris_project/transformers/train_model.py
 create mode 100644 pytest.ini


In [109]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [110]:
import os
import subprocess
import getpass

token = getpass.getpass("Enter your GitHub token: ")

env = os.environ.copy()
env["GITHUB_TOKEN"] = token

askpass_script = """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""

with open("/tmp/git-askpass.sh", "w") as f:
    f.write(askpass_script)

os.chmod("/tmp/git-askpass.sh", 0o700)
env["GIT_ASKPASS"] = "/tmp/git-askpass.sh"
env["GIT_TERMINAL_PROMPT"] = "0"

result = subprocess.run(
    ["git", "push", "origin", "main"],
    env=env,
    text=True
)

print("Push successful!" if result.returncode == 0 else "Push failed.")

Enter your GitHub token: ··········
Push successful!
